# 🔌 Notebook 07: Introduction to MCP (Model Context Protocol)

**Time:** 30 minutes  
**Goal:** Learn to connect LLMs to external tools and data using MCP

## What is MCP?

MCP (Model Context Protocol) is an **open protocol** that enables seamless integration between LLM applications and external data sources and tools.

Think of it as **USB for AI** - a standardized way to connect your LLM to:
- 📁 File systems
- 🔍 Search engines
- 💾 Databases
- 🌐 APIs
- 🛠️ Custom tools

## Why MCP Matters

**Without MCP:**
```
LLM → Hard-coded API calls → Brittle, custom code for each tool
```

**With MCP:**
```
LLM → MCP Protocol → Any MCP Server (modular, reusable)
```

## Key Concepts

### 1. MCP Server
A program that exposes tools/data to LLMs via the MCP protocol.

**Examples:**
- `brave-search` - Web search capabilities
- `filesystem` - File reading/writing
- `github` - Repository operations
- Custom servers you build!

### 2. MCP Client
Your application that connects to MCP servers (that's what we'll build!)

### 3. Tools
Functions that the LLM can call through MCP servers.

**Example tool:**
```json
{
  "name": "search_web",
  "description": "Search the web for information",
  "parameters": {
    "query": "string"
  }
}
```

## What You'll Learn

- Understanding MCP architecture
- Setting up MCP servers
- Calling tools from Python
- Building agent workflows
- Creating your own MCP tools
- Best practices for tool use

Let's build agents that can DO things! 🚀

**Prerequisites:** Notebooks 02-06 completed, Node.js installed (for MCP servers)

In [1]:
# Setup and Imports
import os
import sys
from pathlib import Path
import time
import json
import subprocess
from typing import Dict, List, Any, Optional

# Add parent directory to path
notebook_dir = os.getcwd()
parent_dir = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# Load environment
from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'))

# Import our modules
from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import estimate_tokens, estimate_cost, append_to_reflection
from src.config import PATH

print("=" * 60)
print("NOTEBOOK 07: INTRODUCTION TO MCP")
print("=" * 60)
print()
print(f"Configuration loaded: Path {PATH}")
print()

# Initialize client and tracker
client = LLMClient(path=PATH)
tracker = CostTracker()

# Check Node.js installation (required for MCP servers)
print("Checking prerequisites...")
print("-" * 60)

try:
    node_version = subprocess.check_output(['node', '--version'], 
                                          stderr=subprocess.STDOUT, 
                                          text=True).strip()
    print(f"✓ Node.js installed: {node_version}")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("✗ Node.js NOT found!")
    print("  MCP servers require Node.js")
    print("  Install from: https://nodejs.org/")
    print()
    print("  ⚠️  Some examples in this notebook may not work without Node.js")

try:
    npx_check = subprocess.check_output(['npx', '--version'], 
                                       stderr=subprocess.STDOUT, 
                                       text=True).strip()
    print(f"✓ npx available: {npx_check}")
except (subprocess.CalledProcessError, FileNotFoundError):
    print("✗ npx NOT found (should come with Node.js)")

print()
print("✓ Ready to learn MCP!")
print()

NOTEBOOK 07: INTRODUCTION TO MCP

Configuration loaded: Path A

✓ Claude API client initialized
  Default model: claude-sonnet-4-5-20250929
  Available: Opus 4.5, Sonnet 4.5, Haiku 4.5
Checking prerequisites...
------------------------------------------------------------
✓ Node.js installed: v25.4.0
✓ npx available: 11.7.0

✓ Ready to learn MCP!



---
## 📐 Part 1: MCP Architecture

Let's understand how MCP works before we use it.

In [2]:
# MCP Architecture Explanation
print("=" * 60)
print("MCP ARCHITECTURE")
print("=" * 60)
print()

architecture = """
┌─────────────────────────────────────────────────────────────┐
│                        YOUR APPLICATION                      │
│  ┌─────────────┐         ┌──────────────┐                  │
│  │     LLM     │ ◄─────► │  MCP Client  │                  │
│  │  (Claude)   │         │  (Your Code) │                  │
│  └─────────────┘         └──────────────┘                  │
│                                 │                            │
└─────────────────────────────────┼────────────────────────────┘
                                  │ MCP Protocol
                                  │ (JSON-RPC over stdio)
                ┌─────────────────┼─────────────────┐
                │                 │                 │
        ┌───────▼──────┐  ┌───────▼──────┐  ┌─────▼──────┐
        │ MCP Server 1 │  │ MCP Server 2 │  │ MCP Server │
        │ (brave-search)│  │ (filesystem) │  │  (custom)  │
        └──────────────┘  └──────────────┘  └────────────┘
               │                 │                  │
        ┌──────▼──────┐   ┌──────▼──────┐   ┌──────▼──────┐
        │ Web Search  │   │ Read/Write  │   │ Your Tools  │
        │    API      │   │   Files     │   │             │
        └─────────────┘   └─────────────┘   └─────────────┘

KEY CONCEPTS:

1. MCP Client: Your Python code that manages connections to servers
2. MCP Server: A program that exposes tools via MCP protocol
3. Tool: A function the LLM can call (like "search_web", "read_file")
4. Protocol: Standardized JSON-RPC messages over stdin/stdout

WORKFLOW:
1. LLM decides it needs to use a tool
2. LLM outputs a tool call in JSON format
3. MCP Client sends request to appropriate MCP Server
4. MCP Server executes the tool and returns result
5. Result goes back to LLM
6. LLM continues reasoning with the new information
"""

print(architecture)
print()

print("💡 Key Insight: MCP makes LLMs **agentic** - they can take actions!")
print()

MCP ARCHITECTURE


┌─────────────────────────────────────────────────────────────┐
│                        YOUR APPLICATION                      │
│  ┌─────────────┐         ┌──────────────┐                  │
│  │     LLM     │ ◄─────► │  MCP Client  │                  │
│  │  (Claude)   │         │  (Your Code) │                  │
│  └─────────────┘         └──────────────┘                  │
│                                 │                            │
└─────────────────────────────────┼────────────────────────────┘
                                  │ MCP Protocol
                                  │ (JSON-RPC over stdio)
                ┌─────────────────┼─────────────────┐
                │                 │                 │
        ┌───────▼──────┐  ┌───────▼──────┐  ┌─────▼──────┐
        │ MCP Server 1 │  │ MCP Server 2 │  │ MCP Server │
        │ (brave-search)│  │ (filesystem) │  │  (custom)  │
        └──────────────┘  └──────────────┘  └────────────┘
               │  

---
## 🛠️ Part 2: MCP Server Configuration

MCP servers are configured in Claude Desktop's config file. Let's see how.

In [3]:
# Understanding MCP Configuration
print("=" * 60)
print("MCP SERVER CONFIGURATION")
print("=" * 60)
print()

config_example = """
MCP servers are configured in:
  macOS: ~/Library/Application Support/Claude/claude_desktop_config.json
  Windows: %APPDATA%\\Claude\\claude_desktop_config.json

Example configuration:
{
  "mcpServers": {
    "brave-search": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-brave-search"
      ],
      "env": {
        "BRAVE_API_KEY": "YOUR_BRAVE_API_KEY"
      }
    },
    "filesystem": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        "/Users/username/Desktop"
      ]
    },
    "sequential-thinking": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-sequential-thinking"
      ]
    }
  }
}

CONFIGURATION FIELDS:
- "command": The program to run (usually "npx" for Node-based servers)
- "args": Command-line arguments, including the server package name
- "env": Environment variables (like API keys)

IMPORTANT: After editing config, restart Claude Desktop!
"""

print(config_example)
print()

# Try to detect Claude Desktop config
home = Path.home()
if sys.platform == "darwin":  # macOS
    config_path = home / "Library/Application Support/Claude/claude_desktop_config.json"
elif sys.platform == "win32":  # Windows
    config_path = Path(os.getenv('APPDATA')) / "Claude/claude_desktop_config.json"
else:  # Linux
    config_path = home / ".config/Claude/claude_desktop_config.json"

print("Checking for Claude Desktop config...")
print("-" * 60)
if config_path.exists():
    print(f"✓ Found config at: {config_path}")
    print()
    print("To add MCP servers:")
    print(f"  1. Edit: {config_path}")
    print("  2. Add server configuration (see example above)")
    print("  3. Restart Claude Desktop")
else:
    print(f"✗ Config not found at: {config_path}")
    print()
    print("This is OK if:")
    print("  • You haven't installed Claude Desktop")
    print("  • You're running this in a different environment")
    print()
    print("To use MCP in Claude Desktop:")
    print("  1. Install Claude Desktop from claude.ai")
    print("  2. Create the config file at the path above")
    print("  3. Add your MCP servers to the config")

print()

MCP SERVER CONFIGURATION


MCP servers are configured in:
  macOS: ~/Library/Application Support/Claude/claude_desktop_config.json
  Windows: %APPDATA%\Claude\claude_desktop_config.json

Example configuration:
{
  "mcpServers": {
    "brave-search": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-brave-search"
      ],
      "env": {
        "BRAVE_API_KEY": "YOUR_BRAVE_API_KEY"
      }
    },
    "filesystem": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-filesystem",
        "/Users/username/Desktop"
      ]
    },
    "sequential-thinking": {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-sequential-thinking"
      ]
    }
  }
}

CONFIGURATION FIELDS:
- "command": The program to run (usually "npx" for Node-based servers)
- "args": Command-line arguments, including the server package name
- "env": Environment variables (like API keys)

IMPORTANT: 

---
## 🔍 Part 3: Available MCP Servers

Let's explore some official MCP servers you can use.

In [4]:
# MCP Server Catalog
print("=" * 60)
print("POPULAR MCP SERVERS")
print("=" * 60)
print()

mcp_servers = {
    "Search & Information": {
        "brave-search": {
            "package": "@modelcontextprotocol/server-brave-search",
            "description": "Web search using Brave Search API",
            "requires": "BRAVE_API_KEY",
            "use_case": "Finding current information, research"
        },
        "fetch": {
            "package": "@modelcontextprotocol/server-fetch",
            "description": "Fetch and extract content from URLs",
            "requires": "None",
            "use_case": "Reading web pages, accessing APIs"
        }
    },
    
    "File & Data": {
        "filesystem": {
            "package": "@modelcontextprotocol/server-filesystem",
            "description": "Read/write files in specified directories",
            "requires": "Directory path as argument",
            "use_case": "File management, data processing"
        },
        "sqlite": {
            "package": "@modelcontextprotocol/server-sqlite",
            "description": "Query SQLite databases",
            "requires": "Database path as argument",
            "use_case": "Data analysis, database queries"
        }
    },
    
    "Development": {
        "github": {
            "package": "@modelcontextprotocol/server-github",
            "description": "Interact with GitHub repositories",
            "requires": "GITHUB_PERSONAL_ACCESS_TOKEN",
            "use_case": "Code review, repo management"
        },
        "git": {
            "package": "@modelcontextprotocol/server-git",
            "description": "Git operations on local repositories",
            "requires": "Repository path as argument",
            "use_case": "Version control, commit history"
        }
    },
    
    "Utilities": {
        "sequential-thinking": {
            "package": "@modelcontextprotocol/server-sequential-thinking",
            "description": "Enhanced reasoning through structured thinking",
            "requires": "None",
            "use_case": "Complex problem solving"
        },
        "memory": {
            "package": "@modelcontextprotocol/server-memory",
            "description": "Persistent memory across conversations",
            "requires": "None",
            "use_case": "Maintaining context, user preferences"
        }
    }
}

for category, servers in mcp_servers.items():
    print(f"📁 {category}")
    print("=" * 60)
    
    for name, info in servers.items():
        print(f"\n🔧 {name}")
        print(f"   Package: {info['package']}")
        print(f"   Description: {info['description']}")
        print(f"   Requires: {info['requires']}")
        print(f"   Use Case: {info['use_case']}")
    
    print()

print()
print("💡 Install any server with: npx -y <package-name>")
print("   Full list: https://github.com/modelcontextprotocol/servers")
print()

POPULAR MCP SERVERS

📁 Search & Information

🔧 brave-search
   Package: @modelcontextprotocol/server-brave-search
   Description: Web search using Brave Search API
   Requires: BRAVE_API_KEY
   Use Case: Finding current information, research

🔧 fetch
   Package: @modelcontextprotocol/server-fetch
   Description: Fetch and extract content from URLs
   Requires: None
   Use Case: Reading web pages, accessing APIs

📁 File & Data

🔧 filesystem
   Package: @modelcontextprotocol/server-filesystem
   Description: Read/write files in specified directories
   Requires: Directory path as argument
   Use Case: File management, data processing

🔧 sqlite
   Package: @modelcontextprotocol/server-sqlite
   Description: Query SQLite databases
   Requires: Database path as argument
   Use Case: Data analysis, database queries

📁 Development

🔧 github
   Package: @modelcontextprotocol/server-github
   Description: Interact with GitHub repositories
   Requires: GITHUB_PERSONAL_ACCESS_TOKEN
   Use Case: C

---
## 🧪 Part 4: Tool Use in Practice

Now let's see how LLMs actually use tools through the MCP protocol.

### Tool Calling Flow

When an LLM has access to tools, here's what happens:

**1. User Request:**
```
"Search for recent papers on transformer architectures"
```

**2. LLM Recognizes Need for Tool:**
```json
{
  "type": "tool_use",
  "name": "brave_web_search",
  "input": {
    "query": "transformer architecture papers 2024"
  }
}
```

**3. Tool Executes & Returns Result:**
```json
{
  "type": "tool_result",
  "content": "Found 10 papers: 1. 'Attention is All You Need' ..."
}
```

**4. LLM Continues with Result:**
```
"Based on the search results, here are the most relevant papers..."
```

In [5]:
# Simulating Tool Use
print("=" * 60)
print("EXPERIMENT 1: Understanding Tool Use")
print("=" * 60)
print()

# Simulate what happens when LLM has access to tools
# (In real MCP, tools are automatically available to Claude Desktop)

system_prompt_with_tools = """You are an AI assistant with access to tools.

Available tools:
- search_web(query: str) -> str: Search the web for information
- read_file(path: str) -> str: Read contents of a file
- calculate(expression: str) -> float: Perform calculations

When you need to use a tool, output JSON in this format:
{
  "tool": "tool_name",
  "parameters": {
    "param1": "value1"
  }
}

After using a tool, you'll receive the result and can continue reasoning."""

user_request = """I need to find the latest information about GPT-4's capabilities 
and then calculate how many parameters it has relative to GPT-3."""

print("System Prompt with Tools:")
print("-" * 60)
print(system_prompt_with_tools)
print()

print("User Request:")
print("-" * 60)
print(user_request)
print()

response = client.generate(
    prompt=user_request,
    system=system_prompt_with_tools,
    temperature=0.0,
    max_tokens=300
)

if "error" not in response:
    print("LLM Response:")
    print("=" * 60)
    print(response['content'])
    print("=" * 60)
    tracker.add_call(response)
    
    print()
    print("💡 Notice how the LLM:")
    print("   1. Recognizes it needs external information")
    print("   2. Chooses the right tool (search_web)")
    print("   3. Formats the tool call correctly")
    print("   4. Would continue reasoning after getting results")
else:
    print(f"Error: {response['error']}")

print()

EXPERIMENT 1: Understanding Tool Use

System Prompt with Tools:
------------------------------------------------------------
You are an AI assistant with access to tools.

Available tools:
- search_web(query: str) -> str: Search the web for information
- read_file(path: str) -> str: Read contents of a file
- calculate(expression: str) -> float: Perform calculations

When you need to use a tool, output JSON in this format:
{
  "tool": "tool_name",
  "parameters": {
    "param1": "value1"
  }
}

After using a tool, you'll receive the result and can continue reasoning.

User Request:
------------------------------------------------------------
I need to find the latest information about GPT-4's capabilities 
and then calculate how many parameters it has relative to GPT-3.

LLM Response:
I'll help you find information about GPT-4's capabilities and compare its parameters to GPT-3. Let me start by searching for the latest information.

{
  "tool": "search_web",
  "parameters": {
    "query"

---
## 🔨 Part 5: Building Your First MCP Tool

Let's create a simple custom MCP server with a useful tool.

In [6]:
# Custom MCP Server Example
print("=" * 60)
print("BUILDING A CUSTOM MCP SERVER")
print("=" * 60)
print()

custom_server_code = '''
Example: Simple Research Helper MCP Server

This would be saved as a Node.js file and run as an MCP server.
For this notebook, we'll show the concept (actual implementation 
would be in a separate .js file).

// research-helper-mcp.js
import { Server } from "@modelcontextprotocol/sdk/server/index.js";
import { StdioServerTransport } from "@modelcontextprotocol/sdk/server/stdio.js";

const server = new Server({
  name: "research-helper",
  version: "1.0.0"
});

// Define a tool
server.setRequestHandler("tools/list", async () => {
  return {
    tools: [
      {
        name: "extract_citations",
        description: "Extract citations from a research paper text",
        inputSchema: {
          type: "object",
          properties: {
            text: {
              type: "string",
              description: "The paper text to extract citations from"
            }
          },
          required: ["text"]
        }
      },
      {
        name: "generate_summary",
        description: "Generate a structured summary of a paper",
        inputSchema: {
          type: "object",
          properties: {
            title: { type: "string" },
            abstract: { type: "string" }
          },
          required: ["title", "abstract"]
        }
      }
    ]
  };
});

// Handle tool calls
server.setRequestHandler("tools/call", async (request) => {
  const { name, arguments: args } = request.params;
  
  if (name === "extract_citations") {
    // Simple citation extraction logic
    const citations = args.text.match(/\\[\\d+\\]/g) || [];
    return {
      content: [{
        type: "text",
        text: `Found ${citations.length} citations: ${citations.join(", ")}`
      }]
    };
  }
  
  if (name === "generate_summary") {
    return {
      content: [{
        type: "text",
        text: `Summary for "${args.title}":\\n\\nAbstract: ${args.abstract}`
      }]
    };
  }
});

// Start server
const transport = new StdioServerTransport();
await server.connect(transport);

TO USE THIS SERVER:
1. Save the code above to: research-helper-mcp.js
2. Add to claude_desktop_config.json:
   {
     "mcpServers": {
       "research-helper": {
         "command": "node",
         "args": ["path/to/research-helper-mcp.js"]
       }
     }
   }
3. Restart Claude Desktop
4. The tools will be automatically available!
'''

print(custom_server_code)
print()

print("=" * 60)
print("KEY COMPONENTS OF AN MCP SERVER")
print("=" * 60)
print()

components = {
    "1. Server Setup": "Initialize the MCP server with name and version",
    "2. Tool Definitions": "List available tools with descriptions and schemas",
    "3. Tool Handlers": "Implement the actual logic for each tool",
    "4. Transport": "Communication channel (usually stdio for local servers)",
    "5. Connection": "Start the server and listen for requests"
}

for component, description in components.items():
    print(f"{component}")
    print(f"   {description}")
    print()

print("💡 The MCP SDK handles all the protocol details - you just define tools!")
print()

BUILDING A CUSTOM MCP SERVER


Example: Simple Research Helper MCP Server

This would be saved as a Node.js file and run as an MCP server.
For this notebook, we'll show the concept (actual implementation 
would be in a separate .js file).

// research-helper-mcp.js
import { Server } from "@modelcontextprotocol/sdk/server/index.js";
import { StdioServerTransport } from "@modelcontextprotocol/sdk/server/stdio.js";

const server = new Server({
  name: "research-helper",
  version: "1.0.0"
});

// Define a tool
server.setRequestHandler("tools/list", async () => {
  return {
    tools: [
      {
        name: "extract_citations",
        description: "Extract citations from a research paper text",
        inputSchema: {
          type: "object",
          properties: {
            text: {
              type: "string",
              description: "The paper text to extract citations from"
            }
          },
          required: ["text"]
        }
      },
      {
        name: "generat

---
## 🎯 Your Turn: Practice Tasks

Time to work with MCP concepts!

### 📝 Task 1: Design Your Research Agent Tools

**Goal:** Design the MCP tools your research agent will need.

In [7]:
# TODO - Task 1: Design Your Tools
print("=" * 60)
print("TASK 1: Design Your Research Agent Tools")
print("=" * 60)
print()

# ============================================================================
# TODO: Design 3-5 tools your research agent needs
# ============================================================================

your_tools = """
# YOUR TOOL DESIGNS HERE

For each tool, specify:
1. Name (snake_case)
2. Description (what it does)
3. Parameters (what inputs it needs)
4. Return value (what it outputs)
5. Use case (when your agent would use it)

Example:

Tool 1: search_arxiv
  Description: Search arXiv for research papers
  Parameters:
    - query (string): Search keywords
    - max_results (integer): Maximum papers to return (default: 10)
  Returns: List of paper objects with title, authors, abstract, URL
  Use Case: Initial paper discovery phase

Tool 2: [YOUR TOOL]
  Description: [What it does]
  Parameters:
    - [param1]: [description]
    - [param2]: [description]
  Returns: [What it returns]
  Use Case: [When to use it]

[Continue for 3-5 tools total]
"""

print(your_tools)
print()

# ============================================================================
# TODO: Map tools to your workflow
# ============================================================================

workflow_mapping = """
# MAP TOOLS TO YOUR WORKFLOW

Workflow Step 1: [Description]
  Tools needed: [tool1, tool2]
  Why: [Reasoning]

Workflow Step 2: [Description]
  Tools needed: [tool3]
  Why: [Reasoning]

[Continue for all workflow steps]
"""

print("=" * 60)
print("WORKFLOW MAPPING")
print("=" * 60)
print()
print(workflow_mapping)

# ========================================================================
# TODO: Reflection
# ========================================================================

print()
print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = """
### Tool Design

**Total tools designed:** ___

**Categories:**
- Search/Discovery: ___ tools
- Data Processing: ___ tools
- Analysis: ___ tools
- Output Generation: ___ tools

### Tool Complexity

**Simplest tool:** [name]
[Why is it simple?]

**Most complex tool:** [name]
[What makes it complex?]

### Existing vs Custom

**Tools that exist (can use off-the-shelf):** 
[List them]

**Tools you'd need to build custom:**
[List them]

### Implementation Priority

**Must-have (Priority 1):**
1. [tool name]
2. [tool name]

**Nice-to-have (Priority 2):**
1. [tool name]
2. [tool name]

**Future enhancement (Priority 3):**
1. [tool name]

### Dependencies

**Do your tools depend on each other?**
[Yes/No - explain the dependencies]

**What's the sequence of tool calls?**
[Describe the typical flow]

### Data Flow

**What data flows between tools?**
[Describe how outputs become inputs]

**Any bottlenecks?**
[Potential issues in the flow]

### Error Handling

**What could go wrong with each tool?**
[List potential failure modes]

**How would you handle failures?**
[Your strategy for robustness]

### Real-World Readiness

**Could you actually build these tools?** [Yes/No]

**What skills/resources would you need?**
[Be honest about gaps]

**Timeline to implement:**
- Tool 1: ___ hours/days
- Tool 2: ___ hours/days
[etc.]

### Biggest Insight

[What did you learn about designing tools for agents?]
"""

print(reflection)

append_to_reflection(
    notebook="07",
    section_title="Task 1 - Design Your Research Agent Tools",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 1: Design Your Research Agent Tools


# YOUR TOOL DESIGNS HERE

For each tool, specify:
1. Name (snake_case)
2. Description (what it does)
3. Parameters (what inputs it needs)
4. Return value (what it outputs)
5. Use case (when your agent would use it)

Example:

Tool 1: search_arxiv
  Description: Search arXiv for research papers
  Parameters:
    - query (string): Search keywords
    - max_results (integer): Maximum papers to return (default: 10)
  Returns: List of paper objects with title, authors, abstract, URL
  Use Case: Initial paper discovery phase

Tool 2: [YOUR TOOL]
  Description: [What it does]
  Parameters:
    - [param1]: [description]
    - [param2]: [description]
  Returns: [What it returns]
  Use Case: [When to use it]

[Continue for 3-5 tools total]


WORKFLOW MAPPING


# MAP TOOLS TO YOUR WORKFLOW

Workflow Step 1: [Description]
  Tools needed: [tool1, tool2]
  Why: [Reasoning]

Workflow Step 2: [Description]
  Tools needed: [tool3]
  Why: [Reasoning]

[Continue

### 📝 Task 2: Tool Calling Strategy

**Goal:** Understand when and how your agent should call tools.

In [8]:
# TODO - Task 2: Tool Calling Strategy
print("=" * 60)
print("TASK 2: Tool Calling Strategy")
print("=" * 60)
print()

# Scenario: Design how your agent decides which tools to use
scenario = """
SCENARIO: Your research agent receives this request:

"Find recent papers on transformer architectures, read the top 3, 
extract their key contributions, and create a comparison table."

This requires multiple tool calls in sequence. How should your agent:
1. Decide which tools to use?
2. Determine the order?
3. Know when to stop?
4. Handle errors?
"""

print(scenario)
print()

# ============================================================================
# TODO: Design your tool calling strategy
# ============================================================================

your_strategy = """
# YOUR TOOL CALLING STRATEGY

## Decision Tree

IF user request contains [keywords/pattern]:
  → Use tool: [tool_name]
  → Because: [reasoning]

IF previous tool returned [result_type]:
  → Next tool: [tool_name]
  → Because: [reasoning]

[Continue building your decision logic]

## Example Flow for Scenario

Step 1: [What happens first?]
  Tool: [name]
  Input: [what data?]
  Output: [what's returned?]

Step 2: [What happens next?]
  Tool: [name]
  Input: [from previous step?]
  Output: [what's returned?]

[Continue for all steps]

## Error Handling

IF tool call fails:
  → Action: [retry/skip/fallback?]
  → Reasoning: [why?]

IF tool returns empty results:
  → Action: [what to do?]
  → Reasoning: [why?]

## Stopping Conditions

The agent should stop calling tools when:
1. [Condition 1]
2. [Condition 2]
3. [Condition 3]
"""

print("=" * 60)
print("YOUR STRATEGY")
print("=" * 60)
print(your_strategy)

# ============================================================================
# TODO: Simulate the tool calling sequence
# ============================================================================

print()
print("=" * 60)
print("SIMULATED EXECUTION")
print("=" * 60)
print()

simulation = """
# SIMULATE HOW YOUR AGENT WOULD HANDLE THE SCENARIO

Request: "Find recent papers on transformer architectures, 
         read the top 3, extract their key contributions, 
         and create a comparison table."

Call 1:
  Tool: [name]
  Input: {[parameters]}
  Output: [simulated result]
  Agent's reasoning: [why this tool? what next?]

Call 2:
  Tool: [name]
  Input: {[parameters from previous step]}
  Output: [simulated result]
  Agent's reasoning: [why this tool? what next?]

[Continue for all calls]

Final Output:
  [What the agent produces for the user]

Total Tool Calls: ___
Success: [Yes/No]
"""

print(simulation)

# ========================================================================
# TODO: Reflection
# ========================================================================

print()
print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = """
### Strategy Design

**How many tool calls did your scenario require?** ___

**Could it be done with fewer?** [Yes/No - how?]

**What's the maximum tool calls you'd allow?** ___
[Why this limit?]

### Decision Logic

**How does your agent choose which tool to use?**
[Describe your decision mechanism]

**Is it rule-based or LLM-driven?**
[Explain your approach]

**Advantages of your approach:**
1. [Advantage 1]
2. [Advantage 2]

**Disadvantages:**
1. [Disadvantage 1]
2. [Disadvantage 2]

### Error Resilience

**Most likely point of failure:** [Which step?]

**How would you make it robust?**
[Your error handling strategy]

**Should the agent retry failed calls?** [Yes/No]
[Under what conditions?]

### Efficiency

**Any redundant tool calls in your flow?**
[Identify inefficiencies]

**How could you optimize?**
[Improvement ideas]

**Caching strategy:**
[Should results be cached? For how long?]

### User Experience

**Should users see tool calls happening?** [Yes/No]

**How would you show progress?**
[Progress indicators, messages?]

**What if tools are slow?**
[How to manage user expectations?]

### Production Concerns

**Rate limits:**
[How would you handle API rate limits?]

**Costs:**
[Tool calls add latency and cost - is it worth it?]

**Monitoring:**
[What would you track in production?]

### Biggest Challenge

[What's the hardest part of managing tool calling?]

### Key Insight

[What did you learn about agentic behavior?]
"""

print(reflection)

append_to_reflection(
    notebook="07",
    section_title="Task 2 - Tool Calling Strategy",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 2: Tool Calling Strategy


SCENARIO: Your research agent receives this request:

"Find recent papers on transformer architectures, read the top 3, 
extract their key contributions, and create a comparison table."

This requires multiple tool calls in sequence. How should your agent:
1. Decide which tools to use?
2. Determine the order?
3. Know when to stop?
4. Handle errors?


YOUR STRATEGY

# YOUR TOOL CALLING STRATEGY

## Decision Tree

IF user request contains [keywords/pattern]:
  → Use tool: [tool_name]
  → Because: [reasoning]

IF previous tool returned [result_type]:
  → Next tool: [tool_name]
  → Because: [reasoning]

[Continue building your decision logic]

## Example Flow for Scenario

Step 1: [What happens first?]
  Tool: [name]
  Input: [what data?]
  Output: [what's returned?]

Step 2: [What happens next?]
  Tool: [name]
  Input: [from previous step?]
  Output: [what's returned?]

[Continue for all steps]

## Error Handling

IF tool call fails:
  → Action: [retry/skip

### 📝 Task 3: MCP vs Alternatives

**Goal:** Understand when to use MCP vs other approaches.

In [9]:
# TODO - Task 3: MCP vs Alternatives
print("=" * 60)
print("TASK 3: MCP vs Alternatives")
print("=" * 60)
print()

# Compare different approaches to giving LLMs tools
approaches = {
    "MCP (Model Context Protocol)": {
        "how_it_works": "Standardized protocol, modular servers",
        "pros": [
            "Standardized - reusable across applications",
            "Modular - plug and play servers",
            "Growing ecosystem of servers",
            "Clean separation of concerns"
        ],
        "cons": [
            "Requires MCP infrastructure",
            "Still relatively new",
            "Need to learn MCP SDK"
        ],
        "best_for": "Production apps, reusable tools, Claude Desktop integration"
    },
    
    "Function Calling (API)": {
        "how_it_works": "Define functions in API call, LLM returns function calls",
        "pros": [
            "Built into Claude API",
            "No extra infrastructure",
            "Full control over execution",
            "Well documented"
        ],
        "cons": [
            "Custom code for each tool",
            "Not reusable across apps",
            "Have to handle all execution logic"
        ],
        "best_for": "API-based apps, custom tools, simple workflows"
    },
    
    "Prompt Engineering (Manual)": {
        "how_it_works": "Instruct LLM to output tool calls, parse manually",
        "pros": [
            "Works with any LLM",
            "No special setup",
            "Maximum flexibility"
        ],
        "cons": [
            "Unreliable parsing",
            "More prompt engineering",
            "Error-prone",
            "No validation"
        ],
        "best_for": "Prototyping, simple cases, local models"
    },
    
    "RAG (Retrieval Augmented Generation)": {
        "how_it_works": "Retrieve relevant docs, include in context",
        "pros": [
            "Simple to implement",
            "Good for knowledge retrieval",
            "No tool calling complexity"
        ],
        "cons": [
            "Limited to retrieval",
            "Can't take actions",
            "Context window limits",
            "Not for dynamic tools"
        ],
        "best_for": "Knowledge bases, Q&A, document search"
    }
}

print("COMPARISON OF APPROACHES")
print("=" * 60)
print()

for approach, details in approaches.items():
    print(f"🔧 {approach}")
    print("-" * 60)
    print(f"How it works: {details['how_it_works']}")
    print()
    print("Pros:")
    for pro in details['pros']:
        print(f"  ✓ {pro}")
    print()
    print("Cons:")
    for con in details['cons']:
        print(f"  ✗ {con}")
    print()
    print(f"Best for: {details['best_for']}")
    print()
    print()

# ============================================================================
# TODO: Choose your approach
# ============================================================================

print("=" * 60)
print("YOUR DECISION")
print("=" * 60)
print()

your_decision = """
# CHOOSE YOUR APPROACH FOR YOUR RESEARCH AGENT

For my research agent project, I will use:
[Choose one or a combination]

## Reasoning

Phase 1 (Prototyping):
  Approach: [which one?]
  Why: [reasoning]

Phase 2 (Development):
  Approach: [which one?]
  Why: [reasoning]

Phase 3 (Production - if applicable):
  Approach: [which one?]
  Why: [reasoning]

## Hybrid Approach?

[Will you combine approaches? How?]

## Trade-offs Accepted

[What are you giving up with your choice?]

## Migration Path

[If you start with one approach, how would you migrate to another?]
"""

print(your_decision)

# ========================================================================
# TODO: Reflection
# ========================================================================

print()
print("=" * 60)
print("REFLECTION")
print("=" * 60)
print()

reflection = """
### Approach Selection

**Primary approach:** [MCP/Function Calling/Prompt Engineering/RAG]

**Why this choice?**
[Your detailed reasoning]

### Context Matters

**For rapid prototyping, I'd use:** [approach]
**For production deployment, I'd use:** [approach]
**For academic research, I'd use:** [approach]
**For personal projects, I'd use:** [approach]

[Explain why different contexts need different approaches]

### MCP Specifically

**Will you use MCP in your project?** [Yes/No]

**If Yes:**
- Which MCP servers? [list them]
- Custom servers needed? [Yes/No - which ones?]
- Timeline to implement: [estimate]

**If No:**
- Why not? [reasoning]
- What instead? [your alternative]

### Ecosystem Consideration

**How important is reusability?**
[For your use case, does it matter if tools are reusable?]

**How important is standardization?**
[Does following standards matter for your project?]

### Learning Curve

**How comfortable are you with your chosen approach?** ___/5

**What do you need to learn?**
[Skills/knowledge gaps]

**Learning resources:**
[Where will you learn more?]

### Future-Proofing

**Will your choice scale as your project grows?**
[Yes/No - explain]

**What might force you to change approaches?**
[Potential future pain points]

### Biggest Insight

**Most important factor in choosing an approach:**
[What matters most to you?]

**Biggest surprise:**
[What did you not expect about these approaches?]

### Action Items

**Next steps to implement your chosen approach:**
1. [Step 1]
2. [Step 2]
3. [Step 3]
"""

print(reflection)

append_to_reflection(
    notebook="07",
    section_title="Task 3 - MCP vs Alternatives",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")
print()

TASK 3: MCP vs Alternatives

COMPARISON OF APPROACHES

🔧 MCP (Model Context Protocol)
------------------------------------------------------------
How it works: Standardized protocol, modular servers

Pros:
  ✓ Standardized - reusable across applications
  ✓ Modular - plug and play servers
  ✓ Growing ecosystem of servers
  ✓ Clean separation of concerns

Cons:
  ✗ Requires MCP infrastructure
  ✗ Still relatively new
  ✗ Need to learn MCP SDK

Best for: Production apps, reusable tools, Claude Desktop integration


🔧 Function Calling (API)
------------------------------------------------------------
How it works: Define functions in API call, LLM returns function calls

Pros:
  ✓ Built into Claude API
  ✓ No extra infrastructure
  ✓ Full control over execution
  ✓ Well documented

Cons:
  ✗ Custom code for each tool
  ✗ Not reusable across apps
  ✗ Have to handle all execution logic

Best for: API-based apps, custom tools, simple workflows


🔧 Prompt Engineering (Manual)
---------------

---
## 📚 MCP Resources & Best Practices

In [11]:
# Cell 10: MCP Resources and Best Practices
print("=" * 60)
print("MCP RESOURCES & BEST PRACTICES")
print("=" * 60)
print()

resources = {
    "Official Documentation": [
        "MCP Specification: https://modelcontextprotocol.io/specification/2025-11-25",
        "MCP SDK (TypeScript): https://github.com/modelcontextprotocol/typescript-sdk",
        "MCP SDK (Python): https://github.com/modelcontextprotocol/python-sdk",
        "Official Servers: https://github.com/modelcontextprotocol/servers"
    ],
    
    "Getting Started": [
        "Quickstart Guide: https://modelcontextprotocol.io/quickstart",
        "Building Servers: https://modelcontextprotocol.io/docs/building-servers",
        "Claude Desktop Integration: https://modelcontextprotocol.io/docs/tools/claude-desktop"
    ],
    
    "Community": [
        "Contributing: https://modelcontextprotocol.io/community/contributing",
        "Example Servers: https://github.com/modelcontextprotocol/servers/tree/main/src"
    ]
}

print("📚 LEARNING RESOURCES")
print("=" * 60)
print()

for category, links in resources.items():
    print(f"{category}:")
    for link in links:
        print(f"  • {link}")
    print()

print()
print("=" * 60)
print("BEST PRACTICES")
print("=" * 60)
print()

best_practices = {
    "Tool Design": [
        "✓ Keep tools focused - one tool, one job",
        "✓ Use clear, descriptive names",
        "✓ Document parameters thoroughly",
        "✓ Return structured data when possible",
        "✓ Include error messages in responses",
        "✗ Don't make tools too complex",
        "✗ Don't return unstructured blobs of text"
    ],
    
    "Server Development": [
        "✓ Handle errors gracefully",
        "✓ Validate inputs before processing",
        "✓ Log tool calls for debugging",
        "✓ Make servers stateless when possible",
        "✓ Version your tools",
        "✗ Don't assume perfect inputs",
        "✗ Don't store sensitive data"
    ],
    
    "Agent Design": [
        "✓ Limit maximum tool call depth",
        "✓ Track tool call costs",
        "✓ Show progress to users",
        "✓ Implement timeouts",
        "✓ Have fallback strategies",
        "✗ Don't allow infinite loops",
        "✗ Don't call expensive tools unnecessarily"
    ],
    
    "Security": [
        "✓ Validate all tool inputs",
        "✓ Use environment variables for secrets",
        "✓ Limit file system access",
        "✓ Sanitize user inputs",
        "✓ Implement rate limiting",
        "✗ Don't expose sensitive APIs",
        "✗ Don't trust tool outputs blindly"
    ]
}

for category, practices in best_practices.items():
    print(f"🎯 {category}")
    print("-" * 60)
    for practice in practices:
        print(f"  {practice}")
    print()

print()
print("=" * 60)
print("GOLDEN RULES FOR MCP")
print("=" * 60)
print()
print("1. Tools should be simple, focused, and reliable")
print("2. Always validate inputs and handle errors")
print("3. Make tools reusable across projects")
print("4. Document thoroughly - your future self will thank you")
print("5. Test tools independently before integrating")
print()

MCP RESOURCES & BEST PRACTICES

📚 LEARNING RESOURCES

Official Documentation:
  • MCP Specification: https://modelcontextprotocol.io/specification/2025-11-25
  • MCP SDK (TypeScript): https://github.com/modelcontextprotocol/typescript-sdk
  • MCP SDK (Python): https://github.com/modelcontextprotocol/python-sdk
  • Official Servers: https://github.com/modelcontextprotocol/servers

Getting Started:
  • Quickstart Guide: https://modelcontextprotocol.io/quickstart
  • Building Servers: https://modelcontextprotocol.io/docs/building-servers
  • Claude Desktop Integration: https://modelcontextprotocol.io/docs/tools/claude-desktop

Community:
  • Contributing: https://modelcontextprotocol.io/community/contributing
  • Example Servers: https://github.com/modelcontextprotocol/servers/tree/main/src


BEST PRACTICES

🎯 Tool Design
------------------------------------------------------------
  ✓ Keep tools focused - one tool, one job
  ✓ Use clear, descriptive names
  ✓ Document parameters thorough

---
## ✅ Notebook 07 Complete!

### Summary

You've learned about MCP and agentic tool use! You now know:
- ✅ What MCP is and why it matters
- ✅ MCP architecture and components
- ✅ How to configure MCP servers
- ✅ Available MCP servers you can use
- ✅ How LLMs call tools
- ✅ How to design custom tools
- ✅ Tool calling strategies
- ✅ MCP vs alternative approaches
- ✅ Best practices for tool development

In [12]:
# Final Reflection
print("=" * 60)
print("OVERALL NOTEBOOK REFLECTION")
print("=" * 60)
print()

# ============================================================================
# TODO: Final reflection on MCP
# ============================================================================

reflection = """
### 1. How has MCP changed your view of what LLMs can do?

[Your answer]

### 2. Will you use MCP in your research agent?

[Yes/No - explain your decision]

### 3. Most exciting MCP capability?

[What possibilities excite you most?]

### 4. Biggest concern about agentic systems?

[What worries you about giving LLMs tools?]

### 5. Confidence in building MCP tools? (1-5)

**Confidence:** ___/5

[What would increase it?]

### 6. MCP vs Function Calling - which will you use?

[Your choice and reasoning]

### 7. Key takeaway from this notebook?

[Your main learning]
"""

print(reflection)

# Save reflection
append_to_reflection(
    notebook="07",
    section_title="Overall Reflection",
    reflection_content=reflection,
    output_dir=os.path.join(parent_dir, 'outputs')
)

print()
print("💾 Reflection saved to outputs/homework_reflection.md")

# Show costs
print()
print("=" * 60)
print("YOUR COSTS THIS NOTEBOOK")
print("=" * 60)
print()
tracker.report()

print()
print("=" * 60)
print("✅ NOTEBOOK 07 COMPLETE!")
print("=" * 60)
print()
print("Progress: [████████████████████░] 88% Complete")
print()
print("✓ Notebook 00: Setup Verification")
print("✓ Notebook 01: Environment Setup")
print("✓ Notebook 02: LLM Basics")
print("✓ Notebook 03: CO-STAR Framework")
print("✓ Notebook 04: Structured Outputs")
print("✓ Notebook 05: Chain of Thought")
print("✓ Notebook 06: Model Comparison")
print("✓ Notebook 07: MCP Introduction ← YOU ARE HERE")
print("○ Notebook 08: Project Kickoff")
print()
print("Next: notebooks/08_project_kickoff.ipynb")
print()

OVERALL NOTEBOOK REFLECTION


### 1. How has MCP changed your view of what LLMs can do?

[Your answer]

### 2. Will you use MCP in your research agent?

[Yes/No - explain your decision]

### 3. Most exciting MCP capability?

[What possibilities excite you most?]

### 4. Biggest concern about agentic systems?

[What worries you about giving LLMs tools?]

### 5. Confidence in building MCP tools? (1-5)

**Confidence:** ___/5

[What would increase it?]

### 6. MCP vs Function Calling - which will you use?

[Your choice and reasoning]

### 7. Key takeaway from this notebook?

[Your main learning]


💾 Reflection saved to outputs/homework_reflection.md

YOUR COSTS THIS NOTEBOOK

💰 API COST REPORT
Total API calls: 1
Total input tokens: 166
Total output tokens: 74
Total cost: $0.0016

Recent calls:
  1. [23:30:08] sonnet - 166in/74out - $0.0016

✅ NOTEBOOK 07 COMPLETE!

Progress: [████████████████████░] 88% Complete

✓ Notebook 00: Setup Verification
✓ Notebook 01: Environment Setup
✓ Notebook 